In [1]:
import numpy as np
import sys
sys.path.append("../src/")

In [2]:
from integration import nodes_clenshaw_curtis, npts_clenshaw_curtis, weights_clenshaw_curtis

In [3]:
from interpolation import cheb8, cheb8_deriv, Cheb_coeffs_dct

In [6]:
from IN_integration import IN_Bessel_s2_single_func, IN_Bessel_s2_product_func, IN_Bessel_s2_hybrid_func

In [7]:
from scipy.special import jv
from scipy.integrate import quad

In [12]:
nu1 = 53.5
nu2 = 65.5
omega1 = 2.0
omega2 = 0.2
a = 2.*np.max((nu1/omega1, nu2/omega2))
b = 3*a
f = lambda x: x*np.exp(-x/1000)
fprime = lambda x: np.exp(-x/1000) -0.001*x*np.exp(-x/1000)

In [13]:
# single integration

approx_ans = IN_Bessel_s2_single_func(nu2, omega2, a, b, f(a), f(b), fprime(a), fprime(b))
true_ans = quad(lambda x: f(x)*jv(nu2, omega2*x), a, b, limit=1000000, epsabs=1e-12)[0]
approx_ans/true_ans

1.000016788692863

In [15]:
# double integration with far separated omegas

fa, fb, fpa, fpb = f(a), f(b), fprime(a), fprime(b)

approx_ans = IN_Bessel_s2_product_func(nu1, omega1, nu2, omega2, a, b, fa, fb, fpa, fpb)
true_ans = quad(lambda x: f(x)*jv(nu1, omega1*x)*jv(nu2, omega2*x), a, b, limit=1000000, epsabs=1e-12)[0]
approx_ans/true_ans

1.0000007694151218

In [24]:
# double integration with close omegas

N = 1024
w = weights_clenshaw_curtis(N) * 0.5 * (b-a)
xcc = nodes_clenshaw_curtis(N, a, b)
fx = f(xcc)

approx_ans = IN_Bessel_s2_hybrid_func(nu1, omega1, nu2, omega2*9, a, b, fpa, fpb, xcc[::-1], fx[::-1], w[::-1])
true_ans = quad(lambda x: f(x)*jv(nu1, omega1*x)*jv(nu2, omega2*x*9), a, b, limit=1000000, epsabs=1e-12)[0]
approx_ans/true_ans

1.0000000018688053

In [61]:
def main_integration_Bessel_product(f, nu1, omega1, nu2, omega2, a, b, ncheb=2048):
    """
    Main wrapper to integrate any smooth function f (callable) with two Bessel functions
    f(x) * J_nu1(omega1*x) J_nu2(omega2*x)
    """
    
    # Get Cheb nodes on [-1, 1]
    u = nodes_clenshaw_curtis(ncheb, -1, 1)
    
    # Get Cheb nodes on [a, b]
    xcc = 0.5*(b-a) * u + 0.5*(b+a)
    
    # Get Cheb weights
    w = weights_clenshaw_curtis(ncheb) # for [-1, 1]
    
    # Cheb decomposition of deg 7 of f over the full interval
    c = Cheb_coeffs_dct(f(xcc[::ncheb//8]))
        
    if a >= 2*np.max((nu1/omega1, nu2/omega2)):
        print("Case A")
        fa, fb = cheb8(c, np.array([a, b]), a, b)
        fpa, fpb = cheb8_deriv(c, np.array([a, b]), a, b) # get from endpoints of Cheb expansion
        
        if np.abs(omega1-omega2) >= 1.0:
            print("Case A1")
            integral = IN_Bessel_s2_product_func(nu1, omega1, nu2, omega2, a, b, fa, fb, fpa, fpb)
        else:
            print("Case A2")
            fx = cheb8(c, xcc, a, b)
            integral = IN_Bessel_s2_hybrid_func(nu1, omega1, nu2, omega2, a, b, fpa, fpb, xcc[::-1], fx[::-1], w[::-1])
    
    elif b <= 2*np.min((nu1/omega1, nu2/omega2)): # CC quadrature
        print("Case B")
        f_at_nodes = f(xcc)
        rescaled_weights = w*0.5*(b-a)
        integral = np.dot(rescaled_weights, f_at_nodes)
    
    else:
        print("Case C")
        a_tilde = 2*np.max((nu1/omega1, nu2/omega2))
        rescaled_nodes = 0.5*(a_tilde-a) * u + 0.5*(a_tilde+a)
        rescaled_weights = w*0.5*(a_tilde-a)
        f_at_nodes = cheb8(c, rescaled_nodes, a, b)
        integral = np.dot(rescaled_weights, f_at_nodes)
        fa_tilde, fb = cheb8(c, np.array([a_tilde, b]), a, b)
        fpa_tilde, fpb = cheb8_deriv(c, np.array([a_tilde, b]), a, b)
        integral += IN_Bessel_s2_product_func(nu1, omega1, nu2, omega2, a_tilde, b, fa_tilde, fb, fpa_tilde, fpb)
        
    return integral

In [62]:
approx_ans = main_integration_Bessel_product(f, nu1, omega1, nu2, omega2, a, b, ncheb=2048)
true_ans = quad(lambda x: f(x)*jv(nu1, omega1*x)*jv(nu2, omega2*x), a, b, limit=1000000, epsabs=1e-12)[0]
approx_ans/true_ans

Case A
Case A1


1.0000008024127707

In [63]:
approx_ans = main_integration_Bessel_product(f, nu1, omega1, nu2, 9*omega2, a, b, ncheb=2048)
true_ans = quad(lambda x: f(x)*jv(nu1, omega1*x)*jv(nu2, 9*omega2*x), a, b, limit=1000000, epsabs=1e-12)[0]
approx_ans/true_ans

Case A
Case A2


0.026917293363793397

In [41]:
from numba import jit
from scipy.fft import dct
from rocket_fft import scipy_like

In [42]:
@jit(nopython=True)
def dct_func(x):
    out = dct(x)
    return out

In [45]:
dct_func(np.ones(100))

array([200.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,
         0.])